# Bug hunt: five bugs in a working-looking project

Every cell below runs without an error and prints a plausible number. Each of the five stages contains **one planted mistake**, of the kind the course taught you to spot:
a result that is too good, or too bad, or that changes when it should not. Your job is to find it, say what is wrong, and fix it.

For each stage: read the code, look at the result, and decide what the number should have been. Then write your diagnosis in the markdown cell under it.
The answer key is in `solutions/bug_hunt_answers.ipynb`. It names each bug, fixes it and shows the corrected numbers.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
import torch
from torch import nn

## Stage 1: choosing the useful features

The table has 200 patients and 500 measurements; most are noise. We keep the 10 measurements that relate to the outcome, then evaluate a model.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 500)); y = rng.integers(0, 2, 200)                    # in truth the outcome is unrelated to every measurement
X_best = SelectKBest(f_classif, k=10).fit_transform(X, y)                       # keep the 10 measurements most related to the outcome
X_train, X_test, y_train, y_test = train_test_split(X_best, y, test_size=0.3, random_state=0)
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", round(model.score(X_test, y_test), 3))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 2: when to stop training

We train a small network for 60 epochs and report the test accuracy of the best epoch.

In [ ]:
torch.manual_seed(0); rng = np.random.default_rng(1)
Xm = rng.normal(size=(300, 20)); ym = (Xm[:, 0] + 0.5 * rng.normal(size=300) > 0).astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=0)
Xtr, Xte, ytr, yte = [torch.tensor(a, dtype=torch.float32) for a in (Xtr, Xte, ytr, yte)]
net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1)); opt = torch.optim.Adam(net.parameters(), lr=0.01)
best = 0
for epoch in range(60):
    loss = nn.functional.binary_cross_entropy_with_logits(net(Xtr).squeeze(1), ytr); opt.zero_grad(); loss.backward(); opt.step()
    acc = ((net(Xte).squeeze(1) > 0).float() == yte).float().mean().item()
    best = max(best, acc)                                                      # keep the best epoch
print("reported test accuracy:", round(best, 3))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 3: a categorical column

Customers have a plan (basic, plus, pro). The plan matters a lot for the outcome. We encode it as numbers and fit a model.

In [ ]:
rng = np.random.default_rng(2); n = 600
plan = rng.choice(["basic", "plus", "pro"], n, p=[0.5, 0.3, 0.2]); usage = rng.normal(size=n)
y = ((plan == "pro") * 2.0 + (plan == "basic") * -1.0 + 0.5 * usage + rng.normal(0, 0.5, n) > 0).astype(int)
frame = pd.DataFrame({"plan": plan, "usage": usage})
tr, te, ytr_c, yte_c = train_test_split(frame, y, test_size=0.3, random_state=1)
enc_tr = pd.factorize(tr["plan"])[0]; enc_te = pd.factorize(te["plan"])[0]                                          # encode the plan as 0, 1, 2
model = LogisticRegression().fit(np.c_[enc_tr, tr.usage], ytr_c)
print("test accuracy:", round(model.score(np.c_[enc_te, te.usage], yte_c), 3))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 4: nearest neighbours

We predict who will churn from two features: income in dollars and the number of support calls. We use k-nearest neighbours.

In [ ]:
rng = np.random.default_rng(3); n = 500
income = rng.normal(50_000, 15_000, n); calls = rng.poisson(3, n).astype(float)
churn = (calls + rng.normal(0, 1.0, n) > 4).astype(int)                          # churn depends on the calls, not on the income
Xk = np.c_[income, calls]
Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(Xk, churn, test_size=0.3, random_state=0)
knn = KNeighborsClassifier(n_neighbors=15).fit(Xk_tr, yk_tr)
print("test accuracy:", round(knn.score(Xk_te, yk_te), 3))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 5: predictions on new data

The network below uses dropout. We train it and then evaluate it on the held-out rows.

In [ ]:
torch.manual_seed(0); rng = np.random.default_rng(4)
Xd = rng.normal(size=(400, 10)); yd = (Xd[:, :3].sum(1) > 0).astype(np.float32)
Xd_tr, Xd_te, yd_tr, yd_te = [torch.tensor(a, dtype=torch.float32) for a in train_test_split(Xd, yd, test_size=0.3, random_state=0)]
drop = nn.Sequential(nn.Linear(10, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 1)); opt = torch.optim.Adam(drop.parameters(), lr=0.01)
for epoch in range(150):
    drop.train(); loss = nn.functional.binary_cross_entropy_with_logits(drop(Xd_tr).squeeze(1), yd_tr); opt.zero_grad(); loss.backward(); opt.step()
accs = [((drop(Xd_te).squeeze(1) > 0).float() == yd_te).float().mean().item() for _ in range(5)]
print("test accuracy, five evaluations:", [round(a, 3) for a in accs])

**Your diagnosis:** *(what is wrong, and how would you fix it?)*